# Import Libraries

In [1]:
import gzip
import os
import polars as pl
import requests
import shutil

# Download the Data

In [2]:
TCGA_PROJECTS = [
    'TCGA-ACC', 'TCGA-BLCA', 'TCGA-BRCA', 'TCGA-CESC', 'TCGA-CHOL', 
    'TCGA-COAD', 'TCGA-DLBC', 'TCGA-ESCA', 'TCGA-GBM', 'TCGA-HNSC', 
    'TCGA-KICH', 'TCGA-KIRC', 'TCGA-KIRP', 'TCGA-LAML', 'TCGA-LGG', 
    'TCGA-LIHC', 'TCGA-LUAD', 'TCGA-LUSC', 'TCGA-MESO', 'TCGA-OV', 
    'TCGA-PAAD', 'TCGA-PCPG', 'TCGA-PRAD', 'TCGA-READ', 'TCGA-SARC', 
    'TCGA-SKCM', 'TCGA-STAD', 'TCGA-TGCT', 'TCGA-THCA', 'TCGA-THYM', 
    'TCGA-UCEC', 'TCGA-UCS', 'TCGA-UVM'
]

In [3]:
def download_tcga_data(project, type):
    if type == "methylation":
        tcga_download_endpoint = f'https://gdc-hub.s3.us-east-1.amazonaws.com/download/{project}.methylation450.tsv.gz'
        output_file_path = f'UCSC_TCGA_Data/DNA_Methylation/{project}.gz'
    if type == "gene_expression":
        tcga_download_endpoint = f'https://gdc-hub.s3.us-east-1.amazonaws.com/download/{project}.star_tpm.tsv.gz'
        output_file_path = f'UCSC_TCGA_Data/Gene_Expression/{project}.gz'
    if type == 'protein':
        tcga_download_endpoint = f'https://gdc-hub.s3.us-east-1.amazonaws.com/download/{project}.protein.tsv.gz'
        output_file_path = f'UCSC_TCGA_Data/Protein_Expression/{project}.gz'
    if type == 'clinical':
        tcga_download_endpoint = f'https://gdc-hub.s3.us-east-1.amazonaws.com/download/{project}.clinical.tsv.gz'
        output_file_path = f'UCSC_TCGA_Data/Clinical_Data/{project}.gz'
    if type == 'survival':
        tcga_download_endpoint = f'https://gdc-hub.s3.us-east-1.amazonaws.com/download/{project}.survival.tsv.gz'
        output_file_path = f'UCSC_TCGA_Data/Survival/{project}.gz'

    decompressed_file_path = output_file_path.replace('.gz', '')
    if os.path.exists(decompressed_file_path):
        print(f"File already exists: {decompressed_file_path}")
        return
    
    response = requests.get(tcga_download_endpoint, stream=True)

    if response.status_code == 200:
        print(f"Starting the download for project {project}")
        os.makedirs(os.path.dirname(output_file_path), exist_ok=True)
        with open(output_file_path, 'wb') as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)
        
        print(f"Downloaded file saved to {output_file_path}")
        
        with gzip.open(output_file_path, 'rb') as f_in:
            with open(output_file_path.replace('.gz', ''), 'wb') as f_out:
                shutil.copyfileobj(f_in, f_out)
        
        print(f"Decompressed file saved to {output_file_path.replace('.gz', '')}")

        os.remove(output_file_path)
        print(f"Removed compressed file: {output_file_path}")
    else:
        print(f"Failed to download file. Status code: {response.status_code}")

In [4]:
for project in TCGA_PROJECTS:
    download_tcga_data(project, "methylation")
    download_tcga_data(project, "gene_expression")
    download_tcga_data(project, "protein")

Starting the download for project TCGA-ACC
Downloaded file saved to UCSC_TCGA_Data/DNA_Methylation/TCGA-ACC.gz
Decompressed file saved to UCSC_TCGA_Data/DNA_Methylation/TCGA-ACC
Removed compressed file: UCSC_TCGA_Data/DNA_Methylation/TCGA-ACC.gz
Starting the download for project TCGA-ACC
Downloaded file saved to UCSC_TCGA_Data/Gene_Expression/TCGA-ACC.gz
Decompressed file saved to UCSC_TCGA_Data/Gene_Expression/TCGA-ACC
Removed compressed file: UCSC_TCGA_Data/Gene_Expression/TCGA-ACC.gz
Starting the download for project TCGA-ACC
Downloaded file saved to UCSC_TCGA_Data/Protein_Expression/TCGA-ACC.gz
Decompressed file saved to UCSC_TCGA_Data/Protein_Expression/TCGA-ACC
Removed compressed file: UCSC_TCGA_Data/Protein_Expression/TCGA-ACC.gz
Starting the download for project TCGA-BLCA
Downloaded file saved to UCSC_TCGA_Data/DNA_Methylation/TCGA-BLCA.gz
Decompressed file saved to UCSC_TCGA_Data/DNA_Methylation/TCGA-BLCA
Removed compressed file: UCSC_TCGA_Data/DNA_Methylation/TCGA-BLCA.gz
Sta

# Process the Data

## DNA Methylation

### Determine which probes to keep

In [ ]:
def create_probes_to_remove_set():
    #Downloaded from https://gdc-hub.s3.us-east-1.amazonaws.com/download/HM450.hg38.manifest.gencode.v36.probeMap
    hg38_genes_file_path = 'HM450.hg38.manifest.gencode.v36.tsv'  

    df = pl.read_csv(hg38_genes_file_path, separator='\t',
                     null_values=['NA'])

    selected_columns = df.select(['CpG_chrm', 'probeID', 'genesUniq', 'transcriptTypes'])

    no_gene_associated = set()
    sexual_chr_probes = set()
    chrM_probes = set()
    non_cpg_probes = set()
    non_standard_transcript_probes = set()

    standard_transcript_types = {"protein_coding", "lncRNA", "miRNA"}

    for row in selected_columns.iter_rows(named=True):
        chromosome = row['CpG_chrm']  
        probe_id = str(row['probeID']) 
        gene_id = str(row['genesUniq'])
        gene_names = gene_id.split(";") 
        transcript_types = row['transcriptTypes']
            
        if gene_id == 'None':
            no_gene_associated.add(probe_id)
            continue

        if not probe_id.startswith('cg'):
            non_cpg_probes.add(probe_id)
            continue

        if chromosome == 'chrY' or chromosome=='chrX':
            sexual_chr_probes.add(probe_id)
            continue

        if chromosome=='chrM':
            chrM_probes.add(probe_id)
            continue

        if transcript_types is not None:
            types = transcript_types.split(";")             
            zipped = list(zip(gene_names, types))

            if not any(t in standard_transcript_types for (_, t) in zipped):
                non_standard_transcript_probes.add(probe_id)
                continue

        if len(gene_names) == 1: # Remove AL/AF lncRNA
            if '.' in gene_names[0]:
                non_standard_transcript_probes.add(probe_id)

    print(f"Number of probes without gene: {len(no_gene_associated)}")
    print(f"Number of non standard probes: {len(non_cpg_probes)}")
    print(f"Number of sexual chromosome probes: {len(sexual_chr_probes)}")
    print(f"Number of M chromosome probes: {len(chrM_probes)}")
    print(f"Number of non standard transcript type probes: {len(non_standard_transcript_probes)}")

    probes_to_remove = no_gene_associated.union(non_cpg_probes).union(sexual_chr_probes). \
        union(chrM_probes).union(non_standard_transcript_probes) 

    return probes_to_remove

In [4]:
probes_to_remove = create_probes_to_remove_set()

Number of probes without gene: 61891
Number of non standard probes: 2051
Number of sexual chromosome probes: 10619
Number of M chromosome probes: 6
Number of non standard transcript type probes: 25171


### Remove unwanted probes 

In [5]:
def is_valid_probe(probe_name):
    return probe_name not in probes_to_remove

def process_project_methylation_files(project):
    print(f'Processing data for project {project}')
    project_file_path = f'UCSC_TCGA_Data/DNA_Methylation/{project}'
    output_file_path = ('UCSC_TCGA_Data_Clean/DNA_Methylation/'
        f'{project}_filtered_methylation.tsv')
    
    if os.path.exists(output_file_path):
        return

    df = pl.read_csv(project_file_path, separator='\t',
                     null_values=['NA'])
    print(f'Initial shape {df.shape}')
    df = df.rename({'Composite Element REF': 'Probe'}) 

    df_filtered = df.filter(~df['Probe'].is_in(probes_to_remove))

    os.makedirs(os.path.dirname(output_file_path), exist_ok=True)
    print(df_filtered.shape)
    df_filtered.write_csv(output_file_path, separator='\t')
    print(f'Finished writing data to file {output_file_path}')

    # Remove the original file to save up space  
    #os.remove(project_file_path)
    #print(f"Deleted original file {project_file_path} to free up space.")

In [6]:
for project in TCGA_PROJECTS:
    process_project_methylation_files(project)

Processing data for project TCGA-ACC
Initial shape (486427, 81)
(386689, 81)
Finished writing data to file UCSC_TCGA_Data_Clean/DNA_Methylation/TCGA-ACC_filtered_methylation.tsv
Processing data for project TCGA-BLCA
Initial shape (486427, 438)
(386689, 438)
Finished writing data to file UCSC_TCGA_Data_Clean/DNA_Methylation/TCGA-BLCA_filtered_methylation.tsv
Processing data for project TCGA-BRCA
Initial shape (486427, 894)
(386689, 894)
Finished writing data to file UCSC_TCGA_Data_Clean/DNA_Methylation/TCGA-BRCA_filtered_methylation.tsv
Processing data for project TCGA-CESC
Initial shape (486427, 313)
(386689, 313)
Finished writing data to file UCSC_TCGA_Data_Clean/DNA_Methylation/TCGA-CESC_filtered_methylation.tsv
Processing data for project TCGA-CHOL
Initial shape (486427, 46)
(386689, 46)
Finished writing data to file UCSC_TCGA_Data_Clean/DNA_Methylation/TCGA-CHOL_filtered_methylation.tsv
Processing data for project TCGA-COAD
Initial shape (486427, 347)
(386689, 347)
Finished writing

## Gene Expression

### Determine which genes to keep

In [ ]:
def create_gencode_df():
    # Downloaded from https://www.gencodegenes.org/human/release_36.html
    human_gencode = 'gencode.v36.basic.annotation.gtf'

    human_gencode_df = pl.read_csv(human_gencode, separator='\t',
                        null_values=['NA'], has_header=False, 
                        comment_prefix='#', truncate_ragged_lines=True)

    human_gencode_df = human_gencode_df.rename({
        "column_1": "seqname",
        "column_2": "source",
        "column_3": "feature",
        "column_4": "start",
        "column_5": "end",
        "column_6": "score",
        "column_7": "strand",
        "column_8": "frame",
        "column_9": "attributes",
    })

    human_gencode_df = human_gencode_df.with_columns([
        human_gencode_df["attributes"].str.extract(r'gene_name "([^"]+)"', 1).alias("gene_name"),
        human_gencode_df["attributes"].str.extract(r'gene_type "([^"]+)"', 1).alias("gene_type"),
    ])

    human_gencode_df = human_gencode_df.select(["gene_name", "gene_type"]).unique()
    return human_gencode_df

human_gencode_df = create_gencode_df()

In [ ]:
def create_genes_to_remove_set(human_gencode_df):
    # Downloaded from https://gdc-hub.s3.us-east-1.amazonaws.com/download/gencode.v36.annotation.gtf.gene.probemap
    hg38_genes_file_path = 'gencode.v36.annotation.gtf.gene.probemap' 

    df = pl.read_csv(hg38_genes_file_path, separator='\t',
                     null_values=['NA'])

    selected_columns = df.select(['id', 'gene', 'chrom'])

    sexual_chr_genes = set()
    chrM_probes = set()
    non_standard_genes = set()
    standard_transcript_types = {"protein_coding", "lncRNA", "miRNA"}

    gencode_map = dict(zip(human_gencode_df['gene_name'].to_list(),
                           human_gencode_df['gene_type'].to_list()))

    for row in selected_columns.iter_rows(named=True):
        chromosome = row['chrom']  
        gene_id = str(row['id']) 
        gene_name = str(row['gene'])

        if chromosome == 'chrY' or chromosome == 'chrX':
            sexual_chr_genes.add(gene_id)
            continue

        if chromosome=='chrM':
            chrM_probes.add(gene_id)
            continue

        if gene_name in gencode_map.keys():
            gene_type = gencode_map[gene_name]
            if gene_type not in standard_transcript_types:
                non_standard_genes.add(gene_id)
                continue
        
        if '.' in gene_name:
            non_standard_genes.add(gene_id) # Remove pseudogenes and AL/AF lncRNA

    print(f"Number of non standard genes: {len(non_standard_genes)}")
    print(f"Number of sexual chromosome probes: {len(sexual_chr_genes)}")
    print(f"Number of M chromosome probes: {len(chrM_probes)}")

    ensemblids_to_remove = non_standard_genes.union(sexual_chr_genes).union(chrM_probes)

    return ensemblids_to_remove

In [12]:
ensemblids_to_remove = create_genes_to_remove_set(human_gencode_df)

Number of non standard genes: 33047
Number of sexual chromosome probes: 2990
Number of M chromosome probes: 37


### Remove unwanted genes

In [13]:
def process_project_gene_level_files(project):
    print(f'Processing data for project {project}')
    project_file_path = f'UCSC_TCGA_Data/Gene_Expression/{project}'
    output_file_path = ('UCSC_TCGA_Data_Clean/Gene_Expression/'
        f'{project}_filtered_gene_expression.tsv')
    
    if os.path.exists(output_file_path):
        return

    df = pl.read_csv(project_file_path, separator='\t',
                     null_values=['NA'])
    print(f'Initial shape {df.shape}')
    df = df.rename({'Ensembl_ID': 'Gene'})  

    df_filtered = df.filter(~df['Gene'].is_in(ensemblids_to_remove))
    
    os.makedirs(os.path.dirname(output_file_path), exist_ok=True)
    print(df_filtered.shape)
    df_filtered.write_csv(output_file_path, separator='\t')
    print(f'Finished writing data to file {output_file_path}')

In [14]:
for project in TCGA_PROJECTS:
    process_project_gene_level_files(project)

Processing data for project TCGA-ACC
Initial shape (60660, 80)
(24586, 80)
Finished writing data to file UCSC_TCGA_Data_Clean/Gene_Expression/TCGA-ACC_filtered_gene_expression.tsv
Processing data for project TCGA-BLCA
Initial shape (60660, 429)
(24586, 429)
Finished writing data to file UCSC_TCGA_Data_Clean/Gene_Expression/TCGA-BLCA_filtered_gene_expression.tsv
Processing data for project TCGA-BRCA
Initial shape (60660, 1227)
(24586, 1227)
Finished writing data to file UCSC_TCGA_Data_Clean/Gene_Expression/TCGA-BRCA_filtered_gene_expression.tsv
Processing data for project TCGA-CESC
Initial shape (60660, 310)
(24586, 310)
Finished writing data to file UCSC_TCGA_Data_Clean/Gene_Expression/TCGA-CESC_filtered_gene_expression.tsv
Processing data for project TCGA-CHOL
Initial shape (60660, 45)
(24586, 45)
Finished writing data to file UCSC_TCGA_Data_Clean/Gene_Expression/TCGA-CHOL_filtered_gene_expression.tsv
Processing data for project TCGA-COAD
Initial shape (60660, 515)
(24586, 515)
Finish

## Protein Expression

### Determine which genes to keep

In [ ]:
def create_protein_genes_to_remove_set(ensemblids_to_remove):
    # Downloaded from https://gdc.cancer.gov/about-data/gdc-data-processing/gdc-reference-files
    protein_genes_file_path = 'TCGA_antibodies_descriptions.gencode.v36.tsv'  

    df = pl.read_csv(protein_genes_file_path, separator='\t',
                     null_values=['NA'])
    
    selected_columns = df.select(['peptide_target', 'validation_status', 'gene_id'])

    invalid_antibodies = set()
    ensembl_filtered = set()

    for row in selected_columns.iter_rows(named=True):
        peptide = str(row['peptide_target'])
        gene_id = str(row['gene_id'])
        status = row['validation_status']

        if status != "Valid":
            invalid_antibodies.add(peptide)
            continue
    
        if gene_id in ensemblids_to_remove:
            ensembl_filtered.add(peptide)
    
    print(f"Number of Invalid/Other antibodies: {len(invalid_antibodies)}")
    print(f"Number of Ensembl ID-removed peptides: {len(ensembl_filtered)}")

    peptide_to_remove = invalid_antibodies.union(ensembl_filtered)
    return peptide_to_remove

In [16]:
peptide_to_remove = create_protein_genes_to_remove_set(ensemblids_to_remove)

Number of Invalid/Other antibodies: 204
Number of Ensembl ID-removed peptides: 7


### Remove unwanted peptides

In [17]:
# Doesnt have for LAML project
def process_project_protein_files(project):
    print(f'Processing data for project {project}')
    project_file_path = f'UCSC_TCGA_Data/Protein_Expression/{project}'

    if not os.path.exists(project_file_path):
        print(project_file_path)
        return

    output_file_path = ('UCSC_TCGA_Data_Clean/Protein_Expression/'
        f'{project}_filtered_pe.tsv')
    
    if os.path.exists(output_file_path):
        return

    df = pl.read_csv(project_file_path, separator='\t',
                     null_values=['NA'])
    print(f'Initial shape {df.shape}')

    df_filtered = df.filter(~df['peptide_target'].is_in(peptide_to_remove))
            
    os.makedirs(os.path.dirname(output_file_path), exist_ok=True)
    print(df_filtered.shape)
    df_filtered.write_csv(output_file_path, separator='\t')
    print(f'Finished writing data to file {output_file_path}')

In [18]:
for project in TCGA_PROJECTS:
    process_project_protein_files(project)

Processing data for project TCGA-ACC
Initial shape (487, 47)
(276, 47)
Finished writing data to file UCSC_TCGA_Data_Clean/Protein_Expression/TCGA-ACC_filtered_pe.tsv
Processing data for project TCGA-BLCA
Initial shape (487, 344)
(276, 344)
Finished writing data to file UCSC_TCGA_Data_Clean/Protein_Expression/TCGA-BLCA_filtered_pe.tsv
Processing data for project TCGA-BRCA
Initial shape (487, 920)
(276, 920)
Finished writing data to file UCSC_TCGA_Data_Clean/Protein_Expression/TCGA-BRCA_filtered_pe.tsv
Processing data for project TCGA-CESC
Initial shape (487, 173)
(276, 173)
Finished writing data to file UCSC_TCGA_Data_Clean/Protein_Expression/TCGA-CESC_filtered_pe.tsv
Processing data for project TCGA-CHOL
Initial shape (487, 31)
(276, 31)
Finished writing data to file UCSC_TCGA_Data_Clean/Protein_Expression/TCGA-CHOL_filtered_pe.tsv
Processing data for project TCGA-COAD
Initial shape (487, 364)
(276, 364)
Finished writing data to file UCSC_TCGA_Data_Clean/Protein_Expression/TCGA-COAD_fi

# Intersect the Samples Available for each Omic

In [ ]:
# If downloading the data for the H-VAE
def intersect_project_samples(project):
    print(f'Beginning intersection for {project}')  
    if project == 'TCGA-LAML':
        print('Skipping LAML project for protein expression')
        return 0
    
    dna_methylation_path = f'UCSC_TCGA_Data_Clean/DNA_Methylation/{project}_filtered_methylation.tsv'
    gene_expression_path = f'UCSC_TCGA_Data_Clean/Gene_Expression/{project}_filtered_gene_expression.tsv'
    protein_expression_path = f'UCSC_TCGA_Data_Clean/Protein_Expression/{project}_filtered_pe.tsv'

    dna_df = pl.read_csv(dna_methylation_path, separator='\t')
    ge_df = pl.read_csv(gene_expression_path, separator='\t')
    pe_df = pl.read_csv(protein_expression_path, separator='\t')

    dna_sample_cols = dna_df.select(pl.col(pl.Float64)).columns 
    ge_sample_cols = ge_df.select(pl.col(pl.Float64)).columns  
    pe_sample_cols = pe_df.select(pl.col(pl.Float64)).columns
    
    first_col_dna = dna_df.columns[0]
    first_col_ge = ge_df.columns[0]
    first_col_pe = pe_df.columns[0]
    
    
    common_samples = list(set(dna_sample_cols) & set(ge_sample_cols) & set(pe_sample_cols))
    
    dna_cols_to_keep = [first_col_dna] + common_samples
    ge_cols_to_keep = [first_col_ge] + common_samples
    pe_cols_to_keep = [first_col_pe] + common_samples

    dna_df_filtered = dna_df.select(dna_cols_to_keep)
    ge_df_filtered = ge_df.select(ge_cols_to_keep)
    pe_df_filtered = pe_df.select(pe_cols_to_keep)
    print(f'dna methylation shape: {dna_df_filtered.shape}')
    print(f'gene expression shape: {ge_df_filtered.shape}')
    print(f'protein expression shape: {pe_df_filtered.shape}')

    dna_df_filtered.write_csv(dna_methylation_path, separator='\t')
    ge_df_filtered.write_csv(gene_expression_path, separator='\t')
    pe_df_filtered.write_csv(protein_expression_path, separator='\t')
    print(f'Finished intersecting matrices for project {project}')
    return dna_df_filtered.shape[1]

In [24]:
sample_number = 0
for project in TCGA_PROJECTS:
    count = intersect_project_samples(project)
    sample_number += int(count)
print(f'Total samples: {sample_number}')

Beginning intersection for TCGA-ACC
dna methylation shape: (386689, 47)
gene expression shape: (24586, 47)
protein expression shape: (276, 47)
Finished intersecting matrices for project TCGA-ACC
Beginning intersection for TCGA-BLCA
dna methylation shape: (386689, 339)
gene expression shape: (24586, 339)
protein expression shape: (276, 339)
Finished intersecting matrices for project TCGA-BLCA
Beginning intersection for TCGA-BRCA
dna methylation shape: (386689, 656)
gene expression shape: (24586, 656)
protein expression shape: (276, 656)
Finished intersecting matrices for project TCGA-BRCA
Beginning intersection for TCGA-CESC
dna methylation shape: (386689, 171)
gene expression shape: (24586, 171)
protein expression shape: (276, 171)
Finished intersecting matrices for project TCGA-CESC
Beginning intersection for TCGA-CHOL
dna methylation shape: (386689, 30)
gene expression shape: (24586, 30)
protein expression shape: (276, 30)
Finished intersecting matrices for project TCGA-CHOL
Beginnin

# Verify Sample Type

### Remove samples that are not primary tumor

In [ ]:
def remove_non_primary_tumor_samples(project):
    print(f'Beginning verifying samples type for {project}')  
    if project == 'TCGA-LAML':
        print('Skipping LAML project for protein expression')
        return 0
    
    dna_methylation_path = f'UCSC_TCGA_Data_Clean/DNA_Methylation/{project}_filtered_methylation.tsv'
    gene_expression_path = f'UCSC_TCGA_Data_Clean/Gene_Expression/{project}_filtered_gene_expression.tsv'
    protein_expression_path = f'UCSC_TCGA_Data_Clean/Protein_Expression/{project}_filtered_pe.tsv'

    dna_df = pl.read_csv(dna_methylation_path, separator='\t')
    ge_df = pl.read_csv(gene_expression_path, separator='\t')
    pe_df = pl.read_csv(protein_expression_path, separator='\t')
    print(f'Initial number of samples {dna_df.shape[1]}')

    first_col_dna = dna_df.columns[0]
    first_col_ge = ge_df.columns[0]
    first_col_pe = pe_df.columns[0]

    project_sample_types = {}

    columns_to_keep = []  
    dna_sample_cols = dna_df.select(pl.col(pl.Float64)).columns 
    for column in dna_sample_cols:
        sample_type = column.split('-')[3][:2] 
        if sample_type not in project_sample_types:
            project_sample_types[sample_type] = 1
        else:
            project_sample_types[sample_type] += 1
        if sample_type == '01':   
            columns_to_keep.append(column)

    print(project_sample_types)

    dna_cols_to_keep = [first_col_dna] + columns_to_keep
    ge_cols_to_keep = [first_col_ge] + columns_to_keep
    pe_cols_to_keep = [first_col_pe] + columns_to_keep
    
    dna_df_filtered = dna_df.select(dna_cols_to_keep)
    ge_df_filtered = ge_df.select(ge_cols_to_keep)
    pe_df_filtered = pe_df.select(pe_cols_to_keep)

    dna_df_filtered.write_csv(dna_methylation_path, separator='\t')
    ge_df_filtered.write_csv(gene_expression_path, separator='\t')
    pe_df_filtered.write_csv(protein_expression_path, separator='\t')
    print(f'Final number of samples {dna_df_filtered.shape[1]}')
    print(f'Finished verifying samples type for project {project}')
    return dna_df_filtered.shape[1]

In [ ]:
sample_number = 0
for project in TCGA_PROJECTS:
    count = remove_non_primary_tumor_samples(project)
    sample_number += count
print(f'Total samples: {sample_number}')

# Agreggate Dataframes by Omics

In [ ]:
def aggregate_dataframes_omics(type):
    if type == 'ge':
        directory = 'UCSC_TCGA_Data_Clean/Gene_Expression'
        id_col_name = 'Ensembl_ID'
    elif type == 'pe':
        directory = 'UCSC_TCGA_Data_Clean/Protein_Expression'
        id_col_name = 'AGID'
    elif type == 'dna':
        directory = 'UCSC_TCGA_Data_Clean/DNA_Methylation'
        id_col_name = 'Probe_ID'
    output_file_path = f'UCSC_TCGA_Data_Clean/Aggregated/{type}.tsv'
    all_gene_ids = set()

    for file in os.listdir(directory):
        file_path = os.path.join(directory, file)
        df = pl.read_csv(file_path, separator='\t')
        gene_ids = df.select(df.columns[0]).to_series().to_list()
        all_gene_ids.update(gene_ids)

    aggregated_df = pl.DataFrame({id_col_name: list(all_gene_ids)})

    for file in os.listdir(directory):
        file_path = os.path.join(directory, file)
        df = pl.read_csv(file_path, separator='\t')
        original_id_col = df.columns[0]
        df = df.rename({original_id_col: id_col_name})
        aggregated_df = aggregated_df.join(df, on=id_col_name, how='left', coalesce=True)

    os.makedirs(os.path.dirname(output_file_path), exist_ok=True)
    print(aggregated_df.shape)
    aggregated_df.write_csv(output_file_path, separator='\t')
    print(f'Finished writing aggregated dataframe for {type} type.')

In [14]:
#If working with protein expression, delete LAML project
ge_laml = 'UCSC_TCGA_Data_Clean/Gene_Expression/TCGA-LAML_filtered_gene_expression.tsv'
dna_laml = 'UCSC_TCGA_Data_Clean/DNA_Methylation/TCGA-LAML_filtered_methylation.tsv'

if os.path.exists(ge_laml):
    os.remove(ge_laml)
    print(f'Removed file: {ge_laml}')
if os.path.exists(dna_laml):
    os.remove(dna_laml)
    print(f'Removed file: {dna_laml}')

In [ ]:
# For bigger datasets a lot of swap memory is needed
aggregate_dataframes_omics('ge')
aggregate_dataframes_omics('pe')
aggregate_dataframes_omics('dna')

(24586, 6045)
Finished writing aggregated dataframe for ge type.
(276, 6045)
Finished writing aggregated dataframe for pe type.
(386689, 6045)
Finished writing aggregated dataframe for dna type.


# Remove Unneeded Files

In [1]:
def remove_files_in_directory(dir):
    for file in os.listdir(dir):
        file_path = os.path.join(dir, file)
        os.remove(file_path)

In [4]:
## Free up device space
directories = [
    #'UCSC_TCGA_Data/Gene_Expression',
    #'UCSC_TCGA_Data/DNA_Methylation',
    #'UCSC_TCGA_Data/Protein_Expression',
    'UCSC_TCGA_Data_Clean/Gene_Expression',
    'UCSC_TCGA_Data_Clean/DNA_Methylation',
    'UCSC_TCGA_Data_Clean/Protein_Expression',
    #'UCSC_TCGA_Data_Clean/Aggregated',
]

for dir in directories:
    remove_files_in_directory(dir)

# Normalization

In [16]:
def min_max_scale(type):
    print(f'Beginning normalization for type {type}')

    if type == "gene_expression":
        file_path = 'UCSC_TCGA_Data_Clean/Aggregated/ge.tsv'
        output_file_path = 'UCSC_TCGA_Data_Clean/Aggregated/normalized_ge.tsv'
    elif type == "pe":
        file_path = 'UCSC_TCGA_Data_Clean/Aggregated/pe.tsv'
        output_file_path = 'UCSC_TCGA_Data_Clean/Aggregated/normalized_pe.tsv'

    df = pl.read_csv(file_path, separator='\t')
    print(f'Read file dataframe with shape {df.shape}')
    
    numeric_cols = df.select(pl.col(pl.Float64)).columns

    numeric_df = df.select(numeric_cols)

    min_vals = numeric_df.min_horizontal()
    max_vals = numeric_df.max_horizontal()

    normalized_df = (numeric_df - min_vals) / (max_vals - min_vals)
    result_df = df.with_columns(normalized_df)

    os.makedirs(os.path.dirname(output_file_path), exist_ok=True)
    print(result_df.shape)
    result_df.write_csv(output_file_path, separator='\t')
    print(f'Finished writing data to file {file_path}')

In [17]:
min_max_scale('pe')
min_max_scale('gene_expression')

Beginning normalization for type pe
Read file dataframe with shape (276, 6045)
(276, 6045)
Finished writing data to file UCSC_TCGA_Data_Clean/Aggregated/pe.tsv
Beginning normalization for type gene_expression
Read file dataframe with shape (24586, 6045)
(24586, 6045)
Finished writing data to file UCSC_TCGA_Data_Clean/Aggregated/ge.tsv


# Replace NaN for null

In [38]:
def replace_all_nan():
    files_path = ['UCSC_TCGA_Data_Clean/Aggregated/normalized_ge.tsv', 'UCSC_TCGA_Data_Clean/Aggregated/normalized_pe.tsv', \
                 'UCSC_TCGA_Data_Clean/Aggregated/dna.tsv', 'UCSC_TCGA_Data_Clean/Aggregated/ge.tsv']
    for file_path in files_path:
        if not os.path.exists(file_path):
            continue
        df = pl.read_csv(file_path, separator='\t')
        print((df.null_count().sum_horizontal() > 0).any())
        replaced_nan_df = df.fill_nan(None)
        print((replaced_nan_df.null_count().sum_horizontal() > 0).any())
        replaced_nan_df.write_csv(file_path, separator='\t')

In [ ]:
replace_all_nan()

# Remove Missing Values

In [39]:
def remove_missing_values_threshold(df, var): 
    numeric_cols = df.columns[1:] 
    num_columns = len(numeric_cols)
    na_threshold = num_columns * 0.05
    variance_threshold = var

    if var != 0:
        filtered_df = df.with_columns(
                pl.sum_horizontal([pl.col(c).is_null() for c in numeric_cols])
                    .alias("na_count"),
                pl.concat_list([pl.col(c).cast(pl.Float64) for c in numeric_cols])
                .list.var()
                .alias("row_var")
            )
        
        filtered_df = filtered_df.filter(
                (pl.col("na_count") <= na_threshold) &
                (pl.col("row_var") >= variance_threshold) 
            ).drop(["na_count", "row_var"]) 
    else: 
        filtered_df = df.with_columns(
                pl.sum_horizontal([pl.col(c).is_null() for c in numeric_cols])
                    .alias("na_count")
            )
        
        filtered_df = filtered_df.filter(
                (pl.col("na_count") <= na_threshold)
            ).drop(["na_count"])

    print(filtered_df.shape)
    return filtered_df

In [21]:
def remove_missing_values_methylation():
    directory = 'UCSC_TCGA_Data_Clean/Aggregated'
    output_directory = 'UCSC_TCGA_Data_Clean/Clean/'
    for file in os.listdir(directory):
        if file.startswith('dna'):
            output_file_path = output_directory + file.split('.')[0] + '_clean.tsv'            
            file_path = os.path.join(directory, file)
            df = pl.read_csv(file_path, separator='\t', null_values=['NA'])
            print(f'Read file {file} dataframe with shape {df.shape}')

            filtered_df = remove_missing_values_threshold(df, 0.05)

            os.makedirs(os.path.dirname(output_file_path), exist_ok=True)
            print(filtered_df.shape)
            filtered_df.write_csv(output_file_path, separator='\t')
            print(f'Finished writing data to file {output_file_path}')

In [22]:
remove_missing_values_methylation()

Read file dna.tsv dataframe with shape (386689, 6045)
(29686, 6045)
(29686, 6045)
Finished writing data to file UCSC_TCGA_Data_Clean/Clean/dna_clean.tsv


In [23]:
def remove_missing_and_zero_expression():
    directory = 'UCSC_TCGA_Data_Clean/Aggregated'
    output_directory = 'UCSC_TCGA_Data_Clean/Clean/'
    for file in os.listdir(directory):
        if file.startswith('normalized'):
            output_file_path = output_directory + file.split('.')[0].split('_')[1] + '_clean.tsv'
            file_path = os.path.join(directory, file)
            df = pl.read_csv(file_path, separator='\t', null_values=['NA'])
            print(f'Read file {file} dataframe with shape {df.shape}')

            if not file.startswith('normalized_pe'):
                filtered_df = remove_missing_values_threshold(df, 0)
            else:
                filtered_df = df #redundant

            os.makedirs(os.path.dirname(output_file_path), exist_ok=True)
            print(filtered_df.shape)
            filtered_df.write_csv(output_file_path, separator='\t')
            print(f'Finished writing data to file {output_file_path}')

In [24]:
remove_missing_and_zero_expression()

Read file normalized_pe.tsv dataframe with shape (276, 6045)
(276, 6045)
Finished writing data to file UCSC_TCGA_Data_Clean/Clean/pe_clean.tsv
Read file normalized_ge.tsv dataframe with shape (24586, 6045)
(24073, 6045)
(24073, 6045)
Finished writing data to file UCSC_TCGA_Data_Clean/Clean/ge_clean.tsv


# Imputation

In [40]:
### Used the median to account for skewed data
def impute_row_with_median(df):
    numeric_cols = df.columns[1:] 

    row_medians = (
        df.select(
            pl.concat_list(numeric_cols)  
              .list.drop_nulls()          
              .list.median()              
              .alias("row_median")
        )
        ["row_median"]                   
    )

    df = df.with_columns(
        [
            pl.when(pl.col(c).is_null())
              .then(row_medians)          
              .otherwise(pl.col(c))
              .alias(c)
            for c in numeric_cols
        ]
    )

    return df

def perform_imputation(type):
    print(f'Processing data for type {type}')
    if type == "ge":
        file_path = 'UCSC_TCGA_Data_Clean/Clean/ge_clean.tsv'
    elif type == "pe":
        file_path = 'UCSC_TCGA_Data_Clean/Clean/pe_clean.tsv'
    elif type == 'dna':
        file_path = 'UCSC_TCGA_Data_Clean/Clean/dna_clean.tsv' 

    df = pl.read_csv(file_path, separator='\t')
    resulting_df = impute_row_with_median(df)

    resulting_df.write_csv(file_path, separator='\t')
    print(f'Finished writing data to file {file_path}')

In [26]:
perform_imputation("dna")
perform_imputation("ge")
perform_imputation("pe")

Processing data for type dna
Finished writing data to file UCSC_TCGA_Data_Clean/Clean/dna_clean.tsv
Processing data for type ge
Finished writing data to file UCSC_TCGA_Data_Clean/Clean/ge_clean.tsv
Processing data for type pe
Finished writing data to file UCSC_TCGA_Data_Clean/Clean/pe_clean.tsv


## Create Clinical Data

In [6]:
for project in TCGA_PROJECTS:
    download_tcga_data(project, "clinical")

Starting the download for project TCGA-ACC
Downloaded file saved to UCSC_TCGA_Data/Clinical_Data/TCGA-ACC.gz
Decompressed file saved to UCSC_TCGA_Data/Clinical_Data/TCGA-ACC
Removed compressed file: UCSC_TCGA_Data/Clinical_Data/TCGA-ACC.gz
Starting the download for project TCGA-BLCA
Downloaded file saved to UCSC_TCGA_Data/Clinical_Data/TCGA-BLCA.gz
Decompressed file saved to UCSC_TCGA_Data/Clinical_Data/TCGA-BLCA
Removed compressed file: UCSC_TCGA_Data/Clinical_Data/TCGA-BLCA.gz
Starting the download for project TCGA-BRCA
Downloaded file saved to UCSC_TCGA_Data/Clinical_Data/TCGA-BRCA.gz
Decompressed file saved to UCSC_TCGA_Data/Clinical_Data/TCGA-BRCA
Removed compressed file: UCSC_TCGA_Data/Clinical_Data/TCGA-BRCA.gz
Starting the download for project TCGA-CESC
Downloaded file saved to UCSC_TCGA_Data/Clinical_Data/TCGA-CESC.gz
Decompressed file saved to UCSC_TCGA_Data/Clinical_Data/TCGA-CESC
Removed compressed file: UCSC_TCGA_Data/Clinical_Data/TCGA-CESC.gz
Starting the download for pr

In [28]:
def create_clinical_data_df():
    directory = 'UCSC_TCGA_Data/Clinical_Data/'
    output_file_path = 'UCSC_TCGA_Data_Clean/Clean/clinical.tsv'
    barcode_project_list = []
    if len(os.listdir('UCSC_TCGA_Data_Clean/Clean/')) > 0:
        pe_clean = pl.read_csv('UCSC_TCGA_Data_Clean/Clean/pe_clean.tsv', separator='\t')
        tcga_barcodes = set(pe_clean.columns[1:])
    
    for file in os.listdir(directory):
        file_path = os.path.join(directory, file)
        df = pl.read_csv(file_path, separator='\t', schema_overrides={'submitter_id.annotations': pl.Utf8()},ignore_errors=True)

        selected_columns = df.select(['sample', 'project_id.project', 'tissue_type.samples'])

        for row in selected_columns.iter_rows(named=True):
            sample_code = str(row['sample'])
            project_id = str(row['project_id.project']).split('-')[1]
            sample_type = str(row['tissue_type.samples'])
            if sample_type == 'Normal':
                project_id = 'NORMAL'

            if sample_code not in tcga_barcodes:
                continue
            barcode_project_list.append((sample_code, project_id))

    barcode_project_df = pl.DataFrame(barcode_project_list, schema=["Barcode", "Tumor"], orient='row')

    os.makedirs(os.path.dirname(output_file_path), exist_ok=True)
    barcode_project_df.write_csv(output_file_path, separator='\t')
    return barcode_project_df.shape

In [29]:
df_shape = create_clinical_data_df()
print(df_shape)

(6044, 2)


# Bayesian Networks

In [ ]:
def filter_and_condense_BN(filepath, label, project):
    """Read filtered file, keep only '01' samples, condense IDs, 
       save to BN folder, and delete the original."""
    df = pl.read_csv(filepath, separator='\t')
    first_col = df.columns[0]
    sample_cols = df.select(pl.col(pl.Float64)).columns

    bn_dir = 'UCSC_TCGA_Data_Clean/BN'
    os.makedirs(bn_dir, exist_ok=True)
    condensed_path = os.path.join(bn_dir, f'{project}_{label}_condensed.tsv')

    print(f'[{label}] Initial number of samples: {len(sample_cols)}')

    # Keep only primary tumor samples (type '01')
    columns_to_keep = [col for col in sample_cols if col.split('-')[3][:2] == '01']

    # Deduplicate: keep only the first sample per patient
    seen_patients = set()
    unique_columns = []
    for col in columns_to_keep:
        patient_id = col[:12]
        if patient_id not in seen_patients:
            seen_patients.add(patient_id)
            unique_columns.append(col)
        else:
            print(f'Duplicate patient found: {col} (keeping first occurrence)')

    df_condensed = df.select([first_col] + unique_columns)
    df_condensed = df_condensed.rename({col: col[:12] for col in unique_columns})

    df_condensed.write_csv(condensed_path, separator='\t')
    print(f'  [{label}] Samples after type filter: {len(columns_to_keep)}')
    print(f'  [{label}] Samples after dedup: {len(unique_columns)}')
    print(f'  [{label}] Saved → {condensed_path}')

    # Delete the original filtered file
    # os.remove(filepath)
    # print(f'[{label}] Deleted → {filepath}')

    return len(unique_columns)


def process_project_BN(project):
    print(f'Processing {project}')

    omics = {
        'DNA_Methylation':f'UCSC_TCGA_Data_Clean/DNA_Methylation/{project}_filtered_methylation.tsv',
        'Gene_Expression':f'UCSC_TCGA_Data_Clean/Gene_Expression/{project}_filtered_gene_expression.tsv',
    }

    results = {}
    for label, path in omics.items():
        results[label] = filter_and_condense_BN(path, label, project)

    print(f'Finished processing {project}')
    return results

In [32]:
BNS_PROJECTS = ['TCGA-GBM', 'TCGA-LGG']
sample_number = 0
for project in BNS_PROJECTS:
    results = process_project_BN(project)
    print(results)

Processing TCGA-GBM
  [DNA_Methylation] Initial number of samples: 155
  [DNA_Methylation] Samples after type filter: 140
  [DNA_Methylation] Samples after dedup: 140
  [DNA_Methylation] Saved → UCSC_TCGA_Data_Clean/BN/TCGA-GBM_DNA_Methylation_condensed.tsv
  [Gene_Expression] Initial number of samples: 175
    Duplicate patient found: TCGA-06-0211-01B (keeping first occurrence)
  [Gene_Expression] Samples after type filter: 157
  [Gene_Expression] Samples after dedup: 156
  [Gene_Expression] Saved → UCSC_TCGA_Data_Clean/BN/TCGA-GBM_Gene_Expression_condensed.tsv
  [Protein_Expression] Initial number of samples: 243
  [Protein_Expression] Samples after type filter: 232
  [Protein_Expression] Samples after dedup: 232
  [Protein_Expression] Saved → UCSC_TCGA_Data_Clean/BN/TCGA-GBM_Protein_Expression_condensed.tsv
Finished processing TCGA-GBM
{'DNA_Methylation': 140, 'Gene_Expression': 156, 'Protein_Expression': 232}
Processing TCGA-LGG
  [DNA_Methylation] Initial number of samples: 534
  

In [ ]:
def aggregate_dataframes_omics_BN(omic_type):
    bn_dir = 'UCSC_TCGA_Data_Clean/BN'

    config = {
        'ge':  {'suffix': 'Gene_Expression_condensed.tsv','id_col': 'Ensembl_ID'},
        'dna': {'suffix': 'DNA_Methylation_condensed.tsv','id_col': 'Probe_ID'},
    }

    suffix      = config[omic_type]['suffix']
    id_col_name = config[omic_type]['id_col']
    output_path = os.path.join(bn_dir, f'{omic_type}_aggregated.tsv')

    files = sorted([f for f in os.listdir(bn_dir) if f.endswith(suffix)])
    print(f'Found {len(files)} files for {omic_type}: {files}')

    all_ids = set()
    for file in files:
        file_path = os.path.join(bn_dir, file)
        df = pl.read_csv(file_path, separator='\t')
        ids = df.select(df.columns[0]).to_series().to_list()
        all_ids.update(ids)

    aggregated_df = pl.DataFrame({id_col_name: sorted(list(all_ids))})

    for file in files:
        file_path = os.path.join(bn_dir, file)
        df = pl.read_csv(file_path, separator='\t')
        original_id_col = df.columns[0]
        df = df.rename({original_id_col: id_col_name})
        aggregated_df = aggregated_df.join(df, on=id_col_name, how='left', coalesce=True)
        print(f'  Joined {file} → current shape: {aggregated_df.shape}')

    os.makedirs(bn_dir, exist_ok=True)
    aggregated_df.write_csv(output_path, separator='\t')
    print(f'Final aggregated shape: {aggregated_df.shape}')
    print(f'Saved → {output_path}')
    return aggregated_df.shape

In [ ]:
for omic in ['dna', 'ge']:
    shape = aggregate_dataframes_omics_BN(omic)
    print(shape)

Found 2 files for dna: ['TCGA-GBM_DNA_Methylation_condensed.tsv', 'TCGA-LGG_DNA_Methylation_condensed.tsv']
  Joined TCGA-GBM_DNA_Methylation_condensed.tsv → current shape: (386689, 141)
  Joined TCGA-LGG_DNA_Methylation_condensed.tsv → current shape: (386689, 657)
Final aggregated shape: (386689, 657)
Saved → UCSC_TCGA_Data_Clean/BN/dna_aggregated.tsv
(386689, 657)
Found 2 files for ge: ['TCGA-GBM_Gene_Expression_condensed.tsv', 'TCGA-LGG_Gene_Expression_condensed.tsv']
  Joined TCGA-GBM_Gene_Expression_condensed.tsv → current shape: (24586, 157)
  Joined TCGA-LGG_Gene_Expression_condensed.tsv → current shape: (24586, 673)
Final aggregated shape: (24586, 673)
Saved → UCSC_TCGA_Data_Clean/BN/ge_aggregated.tsv
(24586, 673)
Found 2 files for pe: ['TCGA-GBM_Protein_Expression_condensed.tsv', 'TCGA-LGG_Protein_Expression_condensed.tsv']
  Joined TCGA-GBM_Protein_Expression_condensed.tsv → current shape: (276, 233)
  Joined TCGA-LGG_Protein_Expression_condensed.tsv → current shape: (276, 66

In [35]:
def create_sample_mapping_BN():
    bn_dir = 'UCSC_TCGA_Data_Clean/BN'
    mapping = []

    files = sorted([f for f in os.listdir(bn_dir) if f.endswith('_condensed.tsv')])
    for file in files:
        # Extract project and omic from filename: TCGA-GBM_Gene_Expression_condensed.tsv
        parts = file.replace('_condensed.tsv', '').split('_', 1)
        project = parts[0]          # e.g., TCGA-GBM
        omic = parts[1]             # e.g., Gene_Expression

        file_path = os.path.join(bn_dir, file)
        df = pl.read_csv(file_path, separator='\t')
        sample_cols = [c for c in df.columns if c != df.columns[0]]

        for sample in sample_cols:
            mapping.append({'sample_id': sample, 'project': project, 'omic': omic})

    mapping_df = pl.DataFrame(mapping).unique()
    mapping_path = os.path.join(bn_dir, 'sample_project_mapping.tsv')
    mapping_df.write_csv(mapping_path, separator='\t')
    print(f'Mapping shape: {mapping_df.shape}')
    print(f'Saved → {mapping_path}')
    return mapping_df

In [ ]:
mapping_df = create_sample_mapping_BN()

Mapping shape: (1989, 3)
Saved → UCSC_TCGA_Data_Clean/BN/sample_project_mapping.tsv
shape: (10, 3)
┌──────────────┬──────────┬────────────────────┐
│ sample_id    ┆ project  ┆ omic               │
│ ---          ┆ ---      ┆ ---                │
│ str          ┆ str      ┆ str                │
╞══════════════╪══════════╪════════════════════╡
│ TCGA-E1-A7YL ┆ TCGA-LGG ┆ DNA_Methylation    │
│ TCGA-DB-A64V ┆ TCGA-LGG ┆ DNA_Methylation    │
│ TCGA-HT-7680 ┆ TCGA-LGG ┆ Protein_Expression │
│ TCGA-HT-A614 ┆ TCGA-LGG ┆ DNA_Methylation    │
│ TCGA-FG-5962 ┆ TCGA-LGG ┆ DNA_Methylation    │
│ TCGA-76-4932 ┆ TCGA-GBM ┆ DNA_Methylation    │
│ TCGA-14-0862 ┆ TCGA-GBM ┆ Protein_Expression │
│ TCGA-08-0344 ┆ TCGA-GBM ┆ Protein_Expression │
│ TCGA-P5-A72X ┆ TCGA-LGG ┆ Protein_Expression │
│ TCGA-06-5412 ┆ TCGA-GBM ┆ DNA_Methylation    │
└──────────────┴──────────┴────────────────────┘


In [ ]:
def clean_dna_aggregated_BN():
    bn_dir = 'UCSC_TCGA_Data_Clean/BN'
    input_path = os.path.join(bn_dir, 'dna_aggregated.tsv')
    output_path = os.path.join(bn_dir, 'dna_aggregated_clean.tsv')

    df = pl.read_csv(input_path, separator='\t', null_values=['NA'])
    print(f'Step 1 - Read: {df.shape}')
    df = df.fill_nan(None)
    null_count = df.null_count().sum_horizontal().item(0)
    print(f'Null values after NaN replacement: {null_count}')

    df = remove_missing_values_threshold(df, var=0)
    print(f'Step 2 - After missing/variance filter: {df.shape}')

    df = impute_row_with_median(df)
    remaining_nulls = df.null_count().sum_horizontal().item(0)
    print(f'Step 3 - After imputation, remaining nulls: {remaining_nulls}')

    df.write_csv(output_path, separator='\t')
    print(f'Saved to {output_path}')
    return df

clean_dna_aggregated_BN()

Step 1 - Read: (386689, 657)
  Null values after NaN replacement: 31548171
(323755, 657)
Step 2 - After missing/variance filter: (323755, 657)
Step 3 - After imputation, remaining nulls: 0
Saved → UCSC_TCGA_Data_Clean/BN/dna_aggregated_clean.tsv


Probe_ID,TCGA-76-6192,TCGA-28-2510,TCGA-76-4931,TCGA-76-6656,TCGA-06-0190,TCGA-06-6388,TCGA-4W-AA9S,TCGA-19-5958,TCGA-06-5410,TCGA-28-5220,TCGA-74-6573,TCGA-06-6699,TCGA-19-5960,TCGA-14-0781,TCGA-76-6285,TCGA-81-5910,TCGA-76-6193,TCGA-06-6697,TCGA-28-2501,TCGA-06-6698,TCGA-76-4929,TCGA-76-6282,TCGA-28-5218,TCGA-28-5211,TCGA-32-1980,TCGA-06-5858,TCGA-76-4934,TCGA-06-6693,TCGA-06-6389,TCGA-28-5207,TCGA-19-5950,TCGA-12-5299,TCGA-26-A7UX,TCGA-06-5418,TCGA-74-6575,TCGA-26-5135,…,TCGA-DB-A64R,TCGA-CS-6670,TCGA-S9-A6U1,TCGA-DU-A5TW,TCGA-DB-5270,TCGA-HT-7602,TCGA-IK-8125,TCGA-E1-A7YS,TCGA-HT-7474,TCGA-HT-7693,TCGA-DB-A64P,TCGA-HT-7881,TCGA-HW-7490,TCGA-CS-6186,TCGA-DH-A66G,TCGA-S9-A6WD,TCGA-TM-A84I,TCGA-HT-A74O,TCGA-DB-5273,TCGA-DU-7298,TCGA-FG-5965,TCGA-DU-A5TP,TCGA-DU-7302,TCGA-S9-A7QW,TCGA-P5-A5F0,TCGA-HT-7688,TCGA-HT-8108,TCGA-S9-A7R8,TCGA-S9-A7R1,TCGA-E1-5302,TCGA-FG-A87N,TCGA-HW-A5KK,TCGA-DB-5279,TCGA-DH-A7UV,TCGA-DU-A6S6,TCGA-FG-A711,TCGA-P5-A5EU
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""cg00000029""",0.512752,0.625258,0.699346,0.573895,0.550316,0.195738,0.625146,0.785617,0.263923,0.753366,0.874412,0.67149,0.931495,0.423644,0.478267,0.453448,0.771784,0.448784,0.614201,0.792474,0.841103,0.601932,0.810345,0.492171,0.629527,0.811231,0.376159,0.839457,0.624434,0.409442,0.593762,0.860431,0.910171,0.844036,0.454161,0.770408,…,0.916284,0.848839,0.880445,0.82339,0.830989,0.789014,0.889029,0.913033,0.802345,0.927966,0.900695,0.763284,0.837777,0.746016,0.813709,0.86616,0.601238,0.896194,0.805761,0.832151,0.767935,0.853384,0.908567,0.657379,0.888945,0.608678,0.907689,0.859188,0.543755,0.788435,0.409069,0.744277,0.796582,0.895266,0.824314,0.77225,0.895543
"""cg00000108""",0.967144,0.953088,0.963235,0.943263,0.95732,0.960685,0.963369,0.969406,0.964015,0.968771,0.921722,0.96357,0.962742,0.971464,0.901563,0.932128,0.966196,0.961587,0.962446,0.968386,0.952282,0.95795,0.84494,0.953414,0.957959,0.964857,0.938444,0.94527,0.970267,0.95124,0.964939,0.970368,0.962816,0.962507,0.974587,0.954585,…,0.966052,0.958062,0.964215,0.955347,0.973136,0.962323,0.96134,0.963587,0.973086,0.9669,0.972009,0.967627,0.975538,0.960001,0.960063,0.962681,0.973701,0.962956,0.973565,0.968688,0.973407,0.969753,0.963886,0.967432,0.959887,0.960341,0.972461,0.974109,0.972051,0.971865,0.961053,0.964305,0.974301,0.967127,0.958513,0.9477,0.95965
"""cg00000109""",0.924288,0.90784,0.931319,0.916274,0.867798,0.639612,0.879181,0.832763,0.896851,0.637612,0.905965,0.806154,0.787014,0.913308,0.784834,0.913413,0.694193,0.83589,0.888064,0.921968,0.772994,0.919136,0.91972,0.91906,0.907488,0.770215,0.938189,0.872483,0.919174,0.626166,0.929372,0.935819,0.897519,0.913996,0.917512,0.934975,…,0.926147,0.903376,0.902629,0.910643,0.940333,0.780281,0.875196,0.71481,0.875144,0.915017,0.929786,0.869751,0.948789,0.673219,0.920177,0.847549,0.924826,0.875281,0.891507,0.940054,0.848606,0.930239,0.921044,0.91641,0.905462,0.899195,0.926195,0.901123,0.868061,0.937681,0.912499,0.893274,0.807314,0.926839,0.821398,0.783139,0.915813
"""cg00000236""",0.917945,0.788943,0.921755,0.938134,0.919143,0.916798,0.930672,0.926337,0.847416,0.909716,0.91769,0.928282,0.922496,0.837576,0.90046,0.86196,0.933622,0.921717,0.918668,0.924932,0.906314,0.924884,0.720224,0.871202,0.935838,0.913323,0.898264,0.934572,0.835966,0.91021,0.92634,0.936887,0.925315,0.925476,0.941239,0.939469,…,0.931581,0.941088,0.933861,0.910569,0.930735,0.936314,0.924695,0.926284,0.903677,0.897867,0.935565,0.934113,0.93514,0.943195,0.939734,0.914837,0.897354,0.891676,0.950453,0.953767,0.902765,0.922001,0.944271,0.9331,0.899656,0.931295,0.939439,0.930816,0.938457,0.948374,0.887299,0.917595,0.817016,0.941891,0.899472,0.895159,0.92813
"""cg00000292""",0.588846,0.534096,0.818922,0.758051,0.610711,0.44552

# Survival

In [5]:
SURVIVAL_PROJECTS = ['TCGA-GBM', 'TCGA-LGG']
for project in SURVIVAL_PROJECTS:
    download_tcga_data(project, "survival")

File already exists: UCSC_TCGA_Data/Survival/TCGA-GBM
File already exists: UCSC_TCGA_Data/Survival/TCGA-LGG
